# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant schema
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Record sets, fields, and columns are referenced via their `@id` attributes.

List all record sets and display their fields and types.

In [ ]:
# Show all record sets and their fields by @id
def show_record_sets(schema):
    data = schema.to_json()
    if 'recordSet' not in data:
        print('No record sets defined in dataset schema.')
        return []
    record_sets = data['recordSet']
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    print('Available record sets:')
    for rs in record_sets:
        rid = rs['@id'] if isinstance(rs, dict) else rs
        print(f'  - {rid}')
    print()
    # Now print fields for each record set
    print('Fields in each record set:')
    for rs in record_sets:
        rs_meta = None
        if isinstance(rs, dict):
            rs_meta = rs
        else:
            # Find actual recordset object in metadata
            rs_obj = [obj for obj in data.get('hasPart', []) if isinstance(obj, dict) and obj.get('@id') == rs]
            if rs_obj:
                rs_meta = rs_obj[0]
        if rs_meta is not None:
            rid = rs_meta['@id']
            print(f'  - Record Set: {rid}')
            fields = rs_meta.get('field', [])
            if isinstance(fields, dict):
                fields = [fields]
            for f in fields:
                fid = f['@id'] if isinstance(f, dict) else f
                # Optionally, print type or label if present
                print(f'    * field: {fid}')
    return [rs['@id'] if isinstance(rs, dict) else rs for rs in record_sets]

record_set_ids = show_record_sets(dataset.metadata)

# For demonstration, if no record sets are defined above (record_set_ids is empty),
# try to discover at runtime using dataset API:
if not record_set_ids:
    try:
        # mlcroissant provides dataset.record_sets()
        record_sets = list(dataset.record_sets())
        print('Record Sets (by @id):', record_sets)
        record_set_ids = record_sets
        # Show sample field names by inspecting 1 record from each set
        for rid in record_set_ids:
            print(f'Fields for record set {rid}:')
            try:
                sample = next(dataset.records(record_set=rid))
                print(list(sample.keys()))
            except Exception as e:
                print(f'  Error reading records: {e}')
    except Exception as e:
        print('Could not enumerate record sets:', e)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's extract all records from each record set into pandas DataFrames

# Infer record set IDs if not present
if not record_set_ids:
    record_set_ids = list(dataset.record_sets())

dataframes = {}
for record_set_id in record_set_ids:
    print(f'Loading records for record set {record_set_id}...')
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'  Loaded {len(df)} records, columns: {df.columns.tolist()}')
    except Exception as e:
        print(f'  Failed to load records for {record_set_id}: {e}')

# For demonstration, pick the first loaded record set
if dataframes:
    main_record_set = list(dataframes.keys())[0]
    print(f'Columns in "{main_record_set}":')
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print('No dataframes loaded. Check if the dataset defines record sets and is accessible.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify numeric fields in the main record set for analysis
import numpy as np

if dataframes:
    df = dataframes[main_record_set]
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print('Numeric fields:', numeric_fields)
    if numeric_fields:
        numeric_field = numeric_fields[0]
        # Example threshold for filtering
        threshold = df[numeric_field].quantile(0.75)  # top quartile
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 25%):")
        print(filtered_df.head(3))

        # Normalize the numeric column
        normcol = f"{numeric_field}_normalized"
        filtered_df[normcol] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, normcol]].head())

        # Try grouping by a likely categorical field (e.g. first object/string column)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field} (showing mean {numeric_field}):")
            print(grouped_df.head())
    else:
        print('No numeric fields available for EDA in this record set.')
else:
    print('No data available for EDA analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of a numeric field and a boxplot by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_fields:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' in record set {main_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group (if available)
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored the Mass Spectrometry dataset's structure using `mlcroissant`.
- Loaded and previewed the available record sets by `@id`.
- Performed basic filtering, normalization, and aggregation on available numeric fields.
- Visualized the data to help understand its distribution and major groupings.

This notebook can be extended for downstream analysis such as advanced feature engineering, statistical testing, or machine learning using the structured metadata from the Croissant schema.